# 04: Inteligência de Priorização e Algoritmo Final
Consolidamos as taxas empíricas de performance com os fatores de recência e exclusividade para gerar o ranking de disparo.

**A Nova Fórmula do Score ($S$):**
$$Score = (0.50 \times Taxa\_Sistema) + (0.25 \times Fator\_Recencia) + (0.15 \times Bônus\_DDD) + (0.10 \times Exclusividade)$$

**Justificativa dos pesos:**
- **Taxa_Sistema (0.50):** variável com maior correlação direta com entrega.
- **Fator_Recencia (0.25):** queda clara após 180 dias (módulo 03).
- **Bônus_DDD (0.15):** estabilidade geográfica melhora validade.
- **Exclusividade (0.10):** reduz risco de envio a terceiros (LGPD).

**Regra de Ouro:** Selecionamos os 2 melhores aparelhos distintos por CPF.

In [1]:
import pandas as pd
import numpy as np

df_join = pd.read_parquet('../data/base_performance_mestra.parquet')
taxas_sistemas = pd.read_csv('../data/taxas_empiricas_sistemas.csv').set_index('id_sistema')['taxa_entrega'].to_dict()

# Renomeia cpf da dimensao para cpf_proprietario (schema real dos dados mascarados)
# O parquet base_performance_mestra contem: cpf_y = proprietario real do telefone
if 'cpf_y' in df_join.columns:
    df_join = df_join.rename(columns={'cpf_y': 'cpf_proprietario', 'cpf_x': 'cpf_disparo'})
elif 'cpf' in df_join.columns:
    df_join = df_join.rename(columns={'cpf': 'cpf_proprietario'})

# DDD de referencia (DDD local = moda da base)
ddd_referencia = df_join['telefone_ddd'].mode()[0]

df_join['registro_data_atualizacao'] = pd.to_datetime(df_join['registro_data_atualizacao'], errors='coerce', utc=True)
df_join['envio_datahora'] = pd.to_datetime(df_join['envio_datahora'], errors='coerce', utc=True)
df_join['idade_dado_dias'] = (df_join['envio_datahora'] - df_join['registro_data_atualizacao']).dt.days
print(f'Colunas disponiveis: {list(df_join.columns[:8])}...')
print(f'DDD de referencia: {ddd_referencia}')


Colunas disponiveis: ['id_conta', 'id_hsm', 'id_disparo', 'id_sessao', 'cpf_disparo', 'id_target', 'contato_telefone', 'categoria_hsm']...
DDD de referencia: -1181433720517268842


In [2]:
def fator_recencia(dias):
    if pd.isna(dias) or dias < 0: return 0.5
    if dias <= 30: return 1.0
    if dias <= 180: return 0.8
    if dias <= 365: return 0.5
    return 0.2

# ddd_referencia ja definido na celula anterior


In [3]:
def calcular_score_data_driven(row):
    # Baseado nos achados dos Modulos 02 e 03
    val_fonte = taxas_sistemas.get(row['id_sistema'], 0.20)
    val_rec = fator_recencia(row['idade_dado_dias'])
    val_ddd = 1.0 if row['telefone_ddd'] == ddd_referencia else 0.0
    val_excl = 1.0 if row['telefone_proprietarios_quantidade'] == 1 else 0.0
    
    return (0.50 * val_fonte) + (0.25 * val_rec) + (0.15 * val_ddd) + (0.10 * val_excl)


In [4]:
print(">>> Calculando Scores...")
df_join['score_final'] = df_join.apply(calcular_score_data_driven, axis=1)

>>> Calculando Scores...


In [5]:
# HIGIENIZAÇÃO FINAL (ETAPA CRÍTICA):
# 1. Primeiro garantimos o melhor score para cada par (CPF + Telefone).
# Schema real: cpf_proprietario (da dimensao) = proprietario real do telefone
cpf_col = 'cpf_proprietario' if 'cpf_proprietario' in df_join.columns else 'cpf'

df_clean = df_join.sort_values([cpf_col, 'contato_telefone', 'score_final'], ascending=False)
df_clean = df_clean.drop_duplicates(subset=[cpf_col, 'contato_telefone'])

# 2. AGORA SIM, selecionamos o Top 2 Aparelhos Físicos Distintos por Cidadão.
df_entrega = df_clean.sort_values([cpf_col, 'score_final'], ascending=False).groupby(cpf_col).head(2)


In [6]:
print(f"--- IMPACTO FINAL ---")
cpf_col = 'cpf_proprietario' if 'cpf_proprietario' in df_entrega.columns else 'cpf'
print(f"Cidadãos Únicos: {df_entrega[cpf_col].nunique():,}")
print(f"Mensagens a enviar: {len(df_entrega):,}")
print(f"Economia operacional gerada: {len(df_join) - len(df_entrega):,}")

df_entrega.to_parquet('../data/LISTA_DISPARO_FINAL.parquet')
print(">>> Processo concluído com sucesso.")


--- IMPACTO FINAL ---
Cidadãos Únicos: 881,333
Mensagens a enviar: 902,000
Economia operacional gerada: 5,392,306


>>> Processo concluído com sucesso.
